In [1]:
import pandas as pd
import numpy as np

# Load cleaned Diabetes dataset
diabetes_clean = pd.read_csv("diabetes_clean.csv")

print("Dataset loaded successfully!")
print("Shape:", diabetes_clean.shape)

print("\nColumns:")
print(diabetes_clean.columns.tolist())

Dataset loaded successfully!
Shape: (96146, 9)

Columns:
['gender', 'age', 'hypertension', 'heart_disease', 'smoking_history', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'diabetes']


In [2]:
# Diabetes dataset audit

print("===== DATA TYPES =====")
print(diabetes_clean.dtypes)

print("\n===== MISSING VALUES =====")
print(diabetes_clean.isnull().sum())

print("\n===== DUPLICATE ROWS =====")
print(diabetes_clean.duplicated().sum())

print("\n===== CATEGORICAL VALUES =====")

for col in ["gender", "smoking_history"]:
    print(f"\n{col}:")
    print(diabetes_clean[col].value_counts(dropna=False))

print("\n===== TARGET DISTRIBUTION =====")
print(diabetes_clean["diabetes"].value_counts())

print("\n===== TARGET PERCENTAGE =====")
print(
    diabetes_clean["diabetes"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)


===== DATA TYPES =====
gender                  object
age                    float64
hypertension             int64
heart_disease            int64
smoking_history         object
bmi                    float64
HbA1c_level            float64
blood_glucose_level      int64
diabetes                 int64
dtype: object

===== MISSING VALUES =====
gender                 0
age                    0
hypertension           0
heart_disease          0
smoking_history        0
bmi                    0
HbA1c_level            0
blood_glucose_level    0
diabetes               0
dtype: int64

===== DUPLICATE ROWS =====
0

===== CATEGORICAL VALUES =====

gender:
gender
Female    56161
Male      39967
Other        18
Name: count, dtype: int64

smoking_history:
smoking_history
never          34398
No Info        32887
former          9299
current         9197
not current     6367
ever            3998
Name: count, dtype: int64

===== TARGET DISTRIBUTION =====
diabetes
0    87664
1     8482
Name: count, dty

In [3]:
# ==============================
# DIABETES FEATURE ENGINEERING
# ==============================

diabetes_fe = diabetes_clean.copy()

# 1. Age groups
diabetes_fe["age_group"] = pd.cut(
    diabetes_fe["age"],
    bins=[0, 18, 30, 45, 60, 75, 100],
    labels=[
        "0-18",
        "19-30",
        "31-45",
        "46-60",
        "61-75",
        "76+"
    ],
    include_lowest=True
)

# 2. BMI categories
diabetes_fe["bmi_category"] = pd.cut(
    diabetes_fe["bmi"],
    bins=[0, 18.5, 25, 30, np.inf],
    labels=[
        "Underweight",
        "Normal",
        "Overweight",
        "Obese"
    ],
    include_lowest=True
)

# 3. Glucose × HbA1c interaction
diabetes_fe["glucose_hba1c_interaction"] = (
    diabetes_fe["blood_glucose_level"] *
    diabetes_fe["HbA1c_level"]
)

# 4. Combined cardiovascular indicator
diabetes_fe["cardiovascular_risk"] = (
    (diabetes_fe["hypertension"] == 1) |
    (diabetes_fe["heart_disease"] == 1)
).astype(int)

print("Feature engineering completed!")
print("\nNew shape:", diabetes_fe.shape)

print("\nNew columns:")
print([
    "age_group",
    "bmi_category",
    "glucose_hba1c_interaction",
    "cardiovascular_risk"
])

print("\nAge group distribution:")
print(diabetes_fe["age_group"].value_counts().sort_index())

print("\nBMI category distribution:")
print(diabetes_fe["bmi_category"].value_counts().sort_index())

print("\nCardiovascular risk distribution:")
print(diabetes_fe["cardiovascular_risk"].value_counts())

Feature engineering completed!

New shape: (96146, 13)

New columns:
['age_group', 'bmi_category', 'glucose_hba1c_interaction', 'cardiovascular_risk']

Age group distribution:
age_group
0-18     17820
19-30    14598
31-45    19388
46-60    21735
61-75    14953
76+       7652
Name: count, dtype: int64

BMI category distribution:
bmi_category
Underweight     8518
Normal         22280
Overweight     41871
Obese          23477
Name: count, dtype: int64

Cardiovascular risk distribution:
cardiovascular_risk
0    85677
1    10469
Name: count, dtype: int64


In [4]:
# Separate features and target

X_diabetes = diabetes_fe.drop(columns=["diabetes"])
y_diabetes = diabetes_fe["diabetes"]

print("Features shape:", X_diabetes.shape)
print("Target shape:", y_diabetes.shape)

print("\nTarget distribution:")
print(y_diabetes.value_counts())

print("\nTarget percentage:")
print(
    y_diabetes.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Features shape: (96146, 12)
Target shape: (96146,)

Target distribution:
diabetes
0    87664
1     8482
Name: count, dtype: int64

Target percentage:
diabetes
0    91.18
1     8.82
Name: proportion, dtype: float64


In [5]:
from sklearn.model_selection import train_test_split

X_train_diabetes, X_test_diabetes, y_train_diabetes, y_test_diabetes = train_test_split(
    X_diabetes,
    y_diabetes,
    test_size=0.20,
    stratify=y_diabetes,
    random_state=42
)

print("Training set:", X_train_diabetes.shape)
print("Test set:", X_test_diabetes.shape)

print("\nTraining target distribution:")
print(y_train_diabetes.value_counts())
print(y_train_diabetes.value_counts(normalize=True).mul(100).round(2))

print("\nTest target distribution:")
print(y_test_diabetes.value_counts())
print(y_test_diabetes.value_counts(normalize=True).mul(100).round(2))

Training set: (76916, 12)
Test set: (19230, 12)

Training target distribution:
diabetes
0    70130
1     6786
Name: count, dtype: int64
diabetes
0    91.18
1     8.82
Name: proportion, dtype: float64

Test target distribution:
diabetes
0    17534
1     1696
Name: count, dtype: int64
diabetes
0    91.18
1     8.82
Name: proportion, dtype: float64


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Numerical features
numeric_features_diabetes = [
    "age",
    "hypertension",
    "heart_disease",
    "bmi",
    "HbA1c_level",
    "blood_glucose_level",
    "glucose_hba1c_interaction",
    "cardiovascular_risk"
]

# Categorical features
categorical_features_diabetes = [
    "gender",
    "smoking_history",
    "age_group",
    "bmi_category"
]

# Numerical preprocessing
numeric_transformer_diabetes = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_transformer_diabetes = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        drop="if_binary"
    ))
])

# Combined preprocessing
preprocessor_diabetes = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_diabetes, numeric_features_diabetes),
        ("cat", categorical_transformer_diabetes, categorical_features_diabetes)
    ]
)

print("Diabetes preprocessing pipeline created successfully!")
print("\nNumerical features:", len(numeric_features_diabetes))
print("Categorical features:", len(categorical_features_diabetes))
print("Total input features:", X_train_diabetes.shape[1])

Diabetes preprocessing pipeline created successfully!

Numerical features: 8
Categorical features: 4
Total input features: 12


In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Diabetes Logistic Regression with class weighting
diabetes_lr_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_diabetes),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

print("Diabetes Logistic Regression pipeline created successfully!")
print("Imbalance handling: class_weight='balanced'")

Diabetes Logistic Regression pipeline created successfully!
Imbalance handling: class_weight='balanced'


In [8]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# Train
diabetes_lr_pipeline.fit(
    X_train_diabetes,
    y_train_diabetes
)

# Predictions
y_pred_lr = diabetes_lr_pipeline.predict(X_test_diabetes)

# Probability of diabetes = class 1
y_prob_lr = diabetes_lr_pipeline.predict_proba(X_test_diabetes)[:, 1]

# Evaluation
print("Diabetes Logistic Regression — Test Set")
print("----------------------------------------")
print(f"Accuracy : {accuracy_score(y_test_diabetes, y_pred_lr):.4f}")
print(f"Precision: {precision_score(y_test_diabetes, y_pred_lr):.4f}")
print(f"Recall   : {recall_score(y_test_diabetes, y_pred_lr):.4f}")
print(f"F1 Score : {f1_score(y_test_diabetes, y_pred_lr):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test_diabetes, y_prob_lr):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_diabetes, y_pred_lr))

Diabetes Logistic Regression — Test Set
----------------------------------------
Accuracy : 0.8834
Precision: 0.4227
Recall   : 0.8797
F1 Score : 0.5710
ROC-AUC  : 0.9600

Confusion Matrix:
[[15496  2038]
 [  204  1492]]


In [9]:
from sklearn.neighbors import KNeighborsClassifier

# Diabetes KNN pipeline
diabetes_knn_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_diabetes),
    ("model", KNeighborsClassifier(
        n_neighbors=5,
        weights="distance",
        n_jobs=-1
    ))
])

print("Diabetes KNN pipeline created successfully!")
print("n_neighbors = 5")
print("weights = distance")

Diabetes KNN pipeline created successfully!
n_neighbors = 5
weights = distance


In [10]:
diabetes_knn_pipeline.fit(
    X_train_diabetes,
    y_train_diabetes
)

y_pred_knn = diabetes_knn_pipeline.predict(X_test_diabetes)
y_prob_knn = diabetes_knn_pipeline.predict_proba(X_test_diabetes)[:, 1]

print("Diabetes KNN — Test Set")
print("-----------------------")
print(f"Accuracy : {accuracy_score(y_test_diabetes, y_pred_knn):.4f}")
print(f"Precision: {precision_score(y_test_diabetes, y_pred_knn):.4f}")
print(f"Recall   : {recall_score(y_test_diabetes, y_pred_knn):.4f}")
print(f"F1 Score : {f1_score(y_test_diabetes, y_pred_knn):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test_diabetes, y_prob_knn):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_diabetes, y_pred_knn))

C:\Users\gayat\anaconda3\envs\en1\lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\gayat\anaconda3\envs\en1\lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
  File "C:\Users\gayat\anaconda3\envs\en1\lib\subprocess.py", line 503, in run
    with Popen(*popenargs, **kwargs) as process:
  File "C:\Users\gayat\anaconda3\envs\en1\lib\subprocess.py", line 971, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\gayat\anaconda3\envs\en1\lib\subprocess.py", line 1440, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,


Diabetes KNN — Test Set
-----------------------
Accuracy : 0.9559
Precision: 0.8424
Recall   : 0.6144
F1 Score : 0.7105
ROC-AUC  : 0.8922

Confusion Matrix:
[[17339   195]
 [  654  1042]]


In [11]:
from sklearn.ensemble import RandomForestClassifier

diabetes_rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_diabetes),
    ("model", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

print("Diabetes Random Forest pipeline created successfully!")
print("n_estimators = 300")
print("class_weight = balanced")

Diabetes Random Forest pipeline created successfully!
n_estimators = 300
class_weight = balanced


In [12]:
diabetes_rf_pipeline.fit(
    X_train_diabetes,
    y_train_diabetes
)

y_pred_rf = diabetes_rf_pipeline.predict(X_test_diabetes)
y_prob_rf = diabetes_rf_pipeline.predict_proba(X_test_diabetes)[:, 1]

print("Diabetes Random Forest — Test Set")
print("---------------------------------")
print(f"Accuracy : {accuracy_score(y_test_diabetes, y_pred_rf):.4f}")
print(f"Precision: {precision_score(y_test_diabetes, y_pred_rf):.4f}")
print(f"Recall   : {recall_score(y_test_diabetes, y_pred_rf):.4f}")
print(f"F1 Score : {f1_score(y_test_diabetes, y_pred_rf):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test_diabetes, y_prob_rf):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_diabetes, y_pred_rf))

Diabetes Random Forest — Test Set
---------------------------------
Accuracy : 0.9690
Precision: 0.9379
Recall   : 0.6946
F1 Score : 0.7981
ROC-AUC  : 0.9623

Confusion Matrix:
[[17456    78]
 [  518  1178]]


In [13]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

rf_cv_results = cross_validate(
    diabetes_rf_pipeline,
    X_diabetes,
    y_diabetes,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

print("Diabetes Random Forest — 5-Fold Cross-Validation")
print("------------------------------------------------")
print(f"Accuracy : {rf_cv_results['test_accuracy'].mean():.4f} ± {rf_cv_results['test_accuracy'].std():.4f}")
print(f"Precision: {rf_cv_results['test_precision'].mean():.4f} ± {rf_cv_results['test_precision'].std():.4f}")
print(f"Recall   : {rf_cv_results['test_recall'].mean():.4f} ± {rf_cv_results['test_recall'].std():.4f}")
print(f"F1 Score : {rf_cv_results['test_f1'].mean():.4f} ± {rf_cv_results['test_f1'].std():.4f}")
print(f"ROC-AUC  : {rf_cv_results['test_roc_auc'].mean():.4f} ± {rf_cv_results['test_roc_auc'].std():.4f}")

Diabetes Random Forest — 5-Fold Cross-Validation
------------------------------------------------
Accuracy : 0.9680 ± 0.0007
Precision: 0.9355 ± 0.0053
Recall   : 0.6851 ± 0.0065
F1 Score : 0.7909 ± 0.0051
ROC-AUC  : 0.9670 ± 0.0015


In [14]:
lr_cv_results = cross_validate(
    diabetes_lr_pipeline,
    X_diabetes,
    y_diabetes,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

print("Diabetes Logistic Regression — 5-Fold Cross-Validation")
print("------------------------------------------------------")
print(f"Accuracy : {lr_cv_results['test_accuracy'].mean():.4f} ± {lr_cv_results['test_accuracy'].std():.4f}")
print(f"Precision: {lr_cv_results['test_precision'].mean():.4f} ± {lr_cv_results['test_precision'].std():.4f}")
print(f"Recall   : {lr_cv_results['test_recall'].mean():.4f} ± {lr_cv_results['test_recall'].std():.4f}")
print(f"F1 Score : {lr_cv_results['test_f1'].mean():.4f} ± {lr_cv_results['test_f1'].std():.4f}")
print(f"ROC-AUC  : {lr_cv_results['test_roc_auc'].mean():.4f} ± {lr_cv_results['test_roc_auc'].std():.4f}")

Diabetes Logistic Regression — 5-Fold Cross-Validation
------------------------------------------------------
Accuracy : 0.8861 ± 0.0032
Precision: 0.4294 ± 0.0071
Recall   : 0.8815 ± 0.0061
F1 Score : 0.5774 ± 0.0053
ROC-AUC  : 0.9621 ± 0.0014


In [15]:
from sklearn.ensemble import GradientBoostingClassifier

diabetes_gb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_diabetes),
    ("model", GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ))
])

print("Diabetes Gradient Boosting pipeline created successfully!")
print("n_estimators = 200")
print("learning_rate = 0.05")
print("max_depth = 3")

Diabetes Gradient Boosting pipeline created successfully!
n_estimators = 200
learning_rate = 0.05
max_depth = 3


In [16]:
diabetes_gb_pipeline.fit(
    X_train_diabetes,
    y_train_diabetes
)

y_pred_gb = diabetes_gb_pipeline.predict(X_test_diabetes)
y_prob_gb = diabetes_gb_pipeline.predict_proba(X_test_diabetes)[:, 1]

print("Diabetes Gradient Boosting — Test Set")
print("--------------------------------------")
print(f"Accuracy : {accuracy_score(y_test_diabetes, y_pred_gb):.4f}")
print(f"Precision: {precision_score(y_test_diabetes, y_pred_gb):.4f}")
print(f"Recall   : {recall_score(y_test_diabetes, y_pred_gb):.4f}")
print(f"F1 Score : {f1_score(y_test_diabetes, y_pred_gb):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test_diabetes, y_prob_gb):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_diabetes, y_pred_gb))

Diabetes Gradient Boosting — Test Set
--------------------------------------
Accuracy : 0.9720
Precision: 0.9890
Recall   : 0.6904
F1 Score : 0.8132
ROC-AUC  : 0.9772

Confusion Matrix:
[[17521    13]
 [  525  1171]]


In [17]:
gb_cv_results = cross_validate(
    diabetes_gb_pipeline,
    X_diabetes,
    y_diabetes,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

print("Diabetes Gradient Boosting — 5-Fold Cross-Validation")
print("----------------------------------------------------")
print(f"Accuracy : {gb_cv_results['test_accuracy'].mean():.4f} ± {gb_cv_results['test_accuracy'].std():.4f}")
print(f"Precision: {gb_cv_results['test_precision'].mean():.4f} ± {gb_cv_results['test_precision'].std():.4f}")
print(f"Recall   : {gb_cv_results['test_recall'].mean():.4f} ± {gb_cv_results['test_recall'].std():.4f}")
print(f"F1 Score : {gb_cv_results['test_f1'].mean():.4f} ± {gb_cv_results['test_f1'].std():.4f}")
print(f"ROC-AUC  : {gb_cv_results['test_roc_auc'].mean():.4f} ± {gb_cv_results['test_roc_auc'].std():.4f}")

Diabetes Gradient Boosting — 5-Fold Cross-Validation
----------------------------------------------------
Accuracy : 0.9710 ± 0.0007
Precision: 0.9854 ± 0.0056
Recall   : 0.6811 ± 0.0081
F1 Score : 0.8054 ± 0.0057
ROC-AUC  : 0.9785 ± 0.0011


In [18]:
import joblib

joblib.dump(
    diabetes_gb_pipeline,
    "diabetes_model.pkl"
)

print("Diabetes model saved successfully!")

Diabetes model saved successfully!


In [19]:
import os

print("Exists:", os.path.exists("diabetes_model.pkl"))
print("Size:", round(os.path.getsize("diabetes_model.pkl") / 1024, 2), "KB")

Exists: True
Size: 361.56 KB


In [20]:
import joblib

loaded_diabetes_model = joblib.load("diabetes_model.pkl")

diabetes_test_prediction = loaded_diabetes_model.predict(
    X_test_diabetes.iloc[:5]
)

print("Loaded Diabetes model:", type(loaded_diabetes_model).__name__)
print("Sample predictions:", diabetes_test_prediction)

Loaded Diabetes model: Pipeline
Sample predictions: [0 0 0 0 0]


In [21]:
diabetes_fe.head()

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes,age_group,bmi_category,glucose_hba1c_interaction,cardiovascular_risk
0,Female,80.0,0,1,never,25.19,6.6,140,0,76+,Overweight,924.0,1
1,Female,54.0,0,0,No Info,27.32,6.6,80,0,46-60,Overweight,528.0,0
2,Male,28.0,0,0,never,27.32,5.7,158,0,19-30,Overweight,900.6,0
3,Female,36.0,0,0,current,23.45,5.0,155,0,31-45,Normal,775.0,0
4,Male,76.0,1,1,current,20.14,4.8,155,0,76+,Normal,744.0,1


In [22]:
diabetes_clean.head()

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


In [25]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline

# AdaBoost base estimator
base_tree = DecisionTreeClassifier(
    max_depth=2,
    random_state=42
)

# AdaBoost model
diabetes_adaboost_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_diabetes),
    ("model", AdaBoostClassifier(
        estimator=base_tree,
        n_estimators=100,
        learning_rate=0.05,
        random_state=42
    ))
])

print("Diabetes AdaBoost pipeline created successfully!")
print("n_estimators = 100")
print("learning_rate = 0.05")
print("base tree max_depth = 2")

Diabetes AdaBoost pipeline created successfully!
n_estimators = 100
learning_rate = 0.05
base tree max_depth = 2


In [26]:
# Train AdaBoost
diabetes_adaboost_pipeline.fit(
    X_train_diabetes,
    y_train_diabetes
)

# Predictions
y_pred_ada = diabetes_adaboost_pipeline.predict(
    X_test_diabetes
)

y_prob_ada = diabetes_adaboost_pipeline.predict_proba(
    X_test_diabetes
)[:, 1]

# Evaluation
print("Diabetes AdaBoost — Test Set")
print("----------------------------")
print(f"Accuracy : {accuracy_score(y_test_diabetes, y_pred_ada):.4f}")
print(f"Precision: {precision_score(y_test_diabetes, y_pred_ada):.4f}")
print(f"Recall   : {recall_score(y_test_diabetes, y_pred_ada):.4f}")
print(f"F1 Score : {f1_score(y_test_diabetes, y_pred_ada):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test_diabetes, y_prob_ada):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_diabetes, y_pred_ada))

Diabetes AdaBoost — Test Set
----------------------------
Accuracy : 0.9716
Precision: 1.0000
Recall   : 0.6781
F1 Score : 0.8082
ROC-AUC  : 0.9778

Confusion Matrix:
[[17534     0]
 [  546  1150]]


In [27]:
# AdaBoost — 5-Fold Stratified Cross-Validation

ada_cv_results = cross_validate(
    diabetes_adaboost_pipeline,
    X_diabetes,
    y_diabetes,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

print("Diabetes AdaBoost — 5-Fold Cross-Validation")
print("--------------------------------------------")

print(
    f"Accuracy : "
    f"{ada_cv_results['test_accuracy'].mean():.4f} ± "
    f"{ada_cv_results['test_accuracy'].std():.4f}"
)

print(
    f"Precision: "
    f"{ada_cv_results['test_precision'].mean():.4f} ± "
    f"{ada_cv_results['test_precision'].std():.4f}"
)

print(
    f"Recall   : "
    f"{ada_cv_results['test_recall'].mean():.4f} ± "
    f"{ada_cv_results['test_recall'].std():.4f}"
)

print(
    f"F1 Score : "
    f"{ada_cv_results['test_f1'].mean():.4f} ± "
    f"{ada_cv_results['test_f1'].std():.4f}"
)

print(
    f"ROC-AUC  : "
    f"{ada_cv_results['test_roc_auc'].mean():.4f} ± "
    f"{ada_cv_results['test_roc_auc'].std():.4f}"
)

Diabetes AdaBoost — 5-Fold Cross-Validation
--------------------------------------------
Accuracy : 0.9708 ± 0.0006
Precision: 0.9986 ± 0.0012
Recall   : 0.6705 ± 0.0065
F1 Score : 0.8023 ± 0.0048
ROC-AUC  : 0.9788 ± 0.0010


In [28]:
!pip install xgboost

     -------------------------------------- 101.7/101.7 MB 2.3 MB/s eta 0:00:00


In [29]:
from xgboost import XGBClassifier

# Calculate class imbalance ratio
class_counts = y_train_diabetes.value_counts()

negative_class = class_counts[0]
positive_class = class_counts[1]

scale_pos_weight = negative_class / positive_class

print("Negative class:", negative_class)
print("Positive class:", positive_class)
print("Scale Pos Weight:", round(scale_pos_weight, 4))


# XGBoost model
diabetes_xgb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_diabetes),
    ("model", XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ))
])

print("\nDiabetes XGBoost pipeline created successfully!")

Negative class: 70130
Positive class: 6786
Scale Pos Weight: 10.3345

Diabetes XGBoost pipeline created successfully!


In [30]:
# Train XGBoost
diabetes_xgb_pipeline.fit(
    X_train_diabetes,
    y_train_diabetes
)

# Predictions
y_pred_xgb = diabetes_xgb_pipeline.predict(
    X_test_diabetes
)

y_prob_xgb = diabetes_xgb_pipeline.predict_proba(
    X_test_diabetes
)[:, 1]


# Evaluation
print("Diabetes XGBoost — Test Set")
print("---------------------------")

print(f"Accuracy : {accuracy_score(y_test_diabetes, y_pred_xgb):.4f}")
print(f"Precision: {precision_score(y_test_diabetes, y_pred_xgb):.4f}")
print(f"Recall   : {recall_score(y_test_diabetes, y_pred_xgb):.4f}")
print(f"F1 Score : {f1_score(y_test_diabetes, y_pred_xgb):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test_diabetes, y_prob_xgb):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_diabetes, y_pred_xgb))

Diabetes XGBoost — Test Set
---------------------------
Accuracy : 0.8970
Precision: 0.4583
Recall   : 0.9233
F1 Score : 0.6126
ROC-AUC  : 0.9777

Confusion Matrix:
[[15683  1851]
 [  130  1566]]


In [31]:
# XGBoost — 5-Fold Stratified Cross-Validation

xgb_cv_results = cross_validate(
    diabetes_xgb_pipeline,
    X_diabetes,
    y_diabetes,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

print("Diabetes XGBoost — 5-Fold Cross-Validation")
print("-------------------------------------------")

print(
    f"Accuracy : "
    f"{xgb_cv_results['test_accuracy'].mean():.4f} ± "
    f"{xgb_cv_results['test_accuracy'].std():.4f}"
)

print(
    f"Precision: "
    f"{xgb_cv_results['test_precision'].mean():.4f} ± "
    f"{xgb_cv_results['test_precision'].std():.4f}"
)

print(
    f"Recall   : "
    f"{xgb_cv_results['test_recall'].mean():.4f} ± "
    f"{xgb_cv_results['test_recall'].std():.4f}"
)

print(
    f"F1 Score : "
    f"{xgb_cv_results['test_f1'].mean():.4f} ± "
    f"{xgb_cv_results['test_f1'].std():.4f}"
)

print(
    f"ROC-AUC  : "
    f"{xgb_cv_results['test_roc_auc'].mean():.4f} ± "
    f"{xgb_cv_results['test_roc_auc'].std():.4f}"
)

Diabetes XGBoost — 5-Fold Cross-Validation
-------------------------------------------
Accuracy : 0.9012 ± 0.0039
Precision: 0.4699 ± 0.0101
Recall   : 0.9288 ± 0.0069
F1 Score : 0.6240 ± 0.0084
ROC-AUC  : 0.9791 ± 0.0011


In [32]:
from sklearn.ensemble import VotingClassifier

# Create fresh models for the voting ensemble

voting_lr = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

voting_rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

voting_gb = GradientBoostingClassifier(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

voting_xgb = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)


# Voting Ensemble
voting_model = VotingClassifier(
    estimators=[
        ("lr", voting_lr),
        ("rf", voting_rf),
        ("gb", voting_gb),
        ("xgb", voting_xgb)
    ],
    voting="soft"
)


# Complete pipeline
diabetes_voting_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_diabetes),
    ("model", voting_model)
])

print("Diabetes Voting Ensemble pipeline created successfully!")

Diabetes Voting Ensemble pipeline created successfully!


In [33]:
# Train Voting Ensemble
diabetes_voting_pipeline.fit(
    X_train_diabetes,
    y_train_diabetes
)

# Predictions
y_pred_voting = diabetes_voting_pipeline.predict(
    X_test_diabetes
)

y_prob_voting = diabetes_voting_pipeline.predict_proba(
    X_test_diabetes
)[:, 1]


# Evaluation
print("Diabetes Voting Ensemble — Test Set")
print("------------------------------------")

print(f"Accuracy : {accuracy_score(y_test_diabetes, y_pred_voting):.4f}")
print(f"Precision: {precision_score(y_test_diabetes, y_pred_voting):.4f}")
print(f"Recall   : {recall_score(y_test_diabetes, y_pred_voting):.4f}")
print(f"F1 Score : {f1_score(y_test_diabetes, y_pred_voting):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test_diabetes, y_prob_voting):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_diabetes, y_pred_voting))

Diabetes Voting Ensemble — Test Set
------------------------------------
Accuracy : 0.9606
Precision: 0.7829
Recall   : 0.7653
F1 Score : 0.7740
ROC-AUC  : 0.9746

Confusion Matrix:
[[17174   360]
 [  398  1298]]


In [34]:
# Voting Ensemble — 5-Fold Stratified Cross-Validation

voting_cv_results = cross_validate(
    diabetes_voting_pipeline,
    X_diabetes,
    y_diabetes,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

print("Diabetes Voting Ensemble — 5-Fold Cross-Validation")
print("---------------------------------------------------")

print(
    f"Accuracy : "
    f"{voting_cv_results['test_accuracy'].mean():.4f} ± "
    f"{voting_cv_results['test_accuracy'].std():.4f}"
)

print(
    f"Precision: "
    f"{voting_cv_results['test_precision'].mean():.4f} ± "
    f"{voting_cv_results['test_precision'].std():.4f}"
)

print(
    f"Recall   : "
    f"{voting_cv_results['test_recall'].mean():.4f} ± "
    f"{voting_cv_results['test_recall'].std():.4f}"
)

print(
    f"F1 Score : "
    f"{voting_cv_results['test_f1'].mean():.4f} ± "
    f"{voting_cv_results['test_f1'].std():.4f}"
)

print(
    f"ROC-AUC  : "
    f"{voting_cv_results['test_roc_auc'].mean():.4f} ± "
    f"{voting_cv_results['test_roc_auc'].std():.4f}"
)

Diabetes Voting Ensemble — 5-Fold Cross-Validation
---------------------------------------------------
Accuracy : 0.9613 ± 0.0011
Precision: 0.7883 ± 0.0104
Recall   : 0.7676 ± 0.0065
F1 Score : 0.7778 ± 0.0055
ROC-AUC  : 0.9761 ± 0.0011


In [35]:
from sklearn.ensemble import StackingClassifier

# Base models for stacking

stack_lr = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

stack_rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

stack_gb = GradientBoostingClassifier(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

stack_xgb = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)


# Stacking Ensemble
stacking_model = StackingClassifier(
    estimators=[
        ("lr", stack_lr),
        ("rf", stack_rf),
        ("gb", stack_gb),
        ("xgb", stack_xgb)
    ],
    final_estimator=LogisticRegression(
        max_iter=2000
    ),
    stack_method="predict_proba",
    cv=5,
    n_jobs=-1
)


# Complete pipeline
diabetes_stacking_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_diabetes),
    ("model", stacking_model)
])

print("Diabetes Stacking Ensemble pipeline created successfully!")

Diabetes Stacking Ensemble pipeline created successfully!


In [36]:
# Train Stacking Ensemble

diabetes_stacking_pipeline.fit(
    X_train_diabetes,
    y_train_diabetes
)

# Predictions
y_pred_stacking = diabetes_stacking_pipeline.predict(
    X_test_diabetes
)

y_prob_stacking = diabetes_stacking_pipeline.predict_proba(
    X_test_diabetes
)[:, 1]


# Evaluation
print("Diabetes Stacking Ensemble — Test Set")
print("--------------------------------------")

print(f"Accuracy : {accuracy_score(y_test_diabetes, y_pred_stacking):.4f}")
print(f"Precision: {precision_score(y_test_diabetes, y_pred_stacking):.4f}")
print(f"Recall   : {recall_score(y_test_diabetes, y_pred_stacking):.4f}")
print(f"F1 Score : {f1_score(y_test_diabetes, y_pred_stacking):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test_diabetes, y_prob_stacking):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_diabetes, y_pred_stacking))

Diabetes Stacking Ensemble — Test Set
--------------------------------------
Accuracy : 0.9697
Precision: 0.9219
Recall   : 0.7170
F1 Score : 0.8066
ROC-AUC  : 0.9777

Confusion Matrix:
[[17431   103]
 [  480  1216]]
